In [ ]:
# Import libaries and modules
import pandas as pd
import numpy as np
import sys
import os

print("[i] INFO - Program has been initialised")

[i] INFO - Program has been initialised


In [2]:
# Pre-define variables
overall, rs = 100000, 1
folder, output = 'datasets', 'dataset.csv'
dfs = {}

In [3]:
# Define pre-processing values
proportions = [
    0.40, 0.075, 0.075, 0.075, 0.075,
    0.075, 0.075, 0.075, 0.075
    ]
keep_labels = [
    'Benign', 'DDOS attack-HOIC', 'DDoS attacks-LOIC-HTTP', 'DoS attacks-GoldenEye', 'DoS attacks-Hulk',
    'DoS attacks-Slowloris', 'DoS attacks-SlowHTTPTest', 'FTP-BruteForce', 'SSH-Bruteforce'
    ]
new_labels = [
    'Benign', 'DDoS-HOIC', 'DDoS-LOIC', 'DoS-GoldenEye', 'DoS-Hulk',
    'DoS-Slowloris', 'DoS-SlowHTTPTest', 'FTP-BruteForce', 'SSH-BruteForce'
    ]
cols_to_drop = [
    'Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Timestamp'
    ]
filenames = [
    '02-14-2018.csv', '02-15-2018.csv', '02-16-2018.csv', '02-20-2018.csv', '02-21-2018.csv',
    '02-22-2018.csv', '02-23-2018.csv', '02-28-2018.csv', '03-01-2018.csv', '03-02-2018.csv'
    ]

In [4]:
def load_datasets():
    '''
    Load each dataset from the specified path.

    Parameters:
        None

    Returns:
        None
    '''

    for file in filenames:
        path = os.path.join(folder, file)
        print(f"[i] INFO - Loading '{file}' dataset...")
        try:
            dfs[file] = pd.read_csv(path, low_memory=False)
            print(f"[~] DEBUG - Loaded dataset with shape: ({dfs[file].shape[0]} x {dfs[file].shape[1]})")
        except FileNotFoundError:
            print(f"[!] ERROR - No file found at '{path}'")
            input();sys.exit(1)
        except PermissionError:
            print(f"[!] ERROR - No permission to read '{path}'")
            input();sys.exit(1)
        except UnicodeDecodeError:
            print(f"[!] ERROR - Unable to decode '{path}'")
            input();sys.exit(1)
        except pd.errors.EmptyDataError:
            print(f"[!] ERROR - No data found in '{path}'")
            input();sys.exit(1)
        except pd.errors.ParserError:
            print(f"[!] ERROR - Unable to parse data in '{path}'")
            input();sys.exit(1)

    return

In [5]:
def save_dataset(df_sample):
    '''
    Saves the dataset to the specified path.

    Parameters:
        df_sample

    Returns:
        None
    '''

    path = os.path.join(folder, output)
    print("[i] INFO - Saving dataset...")

    try:
        df_sample.to_csv(path, index=False)
        print(f"[~] DEBUG - Saved dataset with shape: ({df_sample.shape[0]} x {df_sample.shape[1]})")
    except FileNotFoundError:
        print(f"[!] ERROR - No directory found at '{path}'")
        input();sys.exit(1)
    except PermissionError:
        print(f"[!] ERROR - No permission to write to '{path}'")
        input();sys.exit(1)

    return

In [6]:
def process_dataframes():
    '''
    Process each dataframe and store them in a singular dataframe.

    Parameters:
        None

    Returns:
        df_sample
    '''

    # Process each dataframe
    for name, df in dfs.items():
        print(f"[i] INFO - Processing '{name}' dataframe...")

        # Replace inf, -inf and 'Infinity' values
        df = df.replace([np.inf, -np.inf, 'Infinity'], np.nan)

        # Remove missing values
        df = df.dropna()

        # Remove duplicates
        df = df.drop_duplicates()

        # Remove 'Label' values from categories
        df = df[df['Label'] != 'Label']

        # Remove unneeded values from categories
        df = df[df['Label'].isin(keep_labels)]

        # Remove unneeded columns from dataframe
        df = df.drop(columns=cols_to_drop, errors='ignore')

        # Assign a new name to each category
        df['Label'] = df['Label'].replace(dict(zip(keep_labels, new_labels)))

        print(f"[~] DEBUG - New dataset shape: ({df.shape[0]} x {df.shape[1]})")

        dfs[name] = df

    # Combine each dataframe into a singular dataframe
    print("[i] INFO - Combining dataframes into a singular dataframe...")
    df_all = pd.concat(dfs.values(), ignore_index=True)

    # Sample data using overall and proportions value
    counts = [int(overall * p) for p in proportions]
    target_counts = dict(zip(new_labels, counts))

    df_sample = (
        df_all.groupby('Label', group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), target_counts[x.name]), random_state=rs).assign(Label=x.name))
        .reset_index(drop=True)
    )

    return df_sample

In [7]:
def main():
    '''
    Process each dataset.

    Parameters:
        None

    Returns:
        None
    '''

    # Load the datasets
    load_datasets()

    # Perform pre-processing on the datasets
    df_sample = process_dataframes()

    # Save the dataset to a .csv file
    save_dataset(df_sample)

    print("[i] INFO - Program has finished execution")

if __name__ == "__main__":
    main()

[i] INFO - Loading '02-14-2018.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (1048575 x 80)
[i] INFO - Loading '02-15-2018.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (1048575 x 80)
[i] INFO - Loading '02-16-2018.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (1048575 x 80)
[i] INFO - Loading '02-20-2018.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (7948748 x 84)
[i] INFO - Loading '02-21-2018.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (1048575 x 80)
[i] INFO - Loading '02-22-2018.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (1048575 x 80)
[i] INFO - Loading '02-23-2018.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (1048575 x 80)
[i] INFO - Loading '02-28-2018.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (613104 x 80)
[i] INFO - Loading '03-01-2018.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (331125 x 80)
[i] INFO - Loading '03-02-2018.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (104857